The same example as for Qiskit.

In [11]:
import pennylane as qml
import numpy as np
from scipy.optimize import minimize

In [12]:
# Graph
n = 3
edges = [
    (0, 1, 1.0),
    (1, 2, 2.0),
    (0, 2, 3.0)
]

In [13]:
# Cost Hamiltonian
# H_C = Z0 Z1 + 2 Z1 Z2 + 3 Z0 Z2
H_C = qml.dot(
    [w for i, j, w in edges],
    [qml.Z(i) @ qml.Z(j) for i, j, w in edges]
)

In [14]:
# Initialize the simulator
dev = qml.device("default.qubit", wires=n)

In [15]:
# QAOA Ansatz
def ansatz(gammas, betas):

    # Initial state
    for i in range(n):
        qml.Hadamard(wires=i)

    # Apply p layers of QAOA
    for gamma, beta in zip(gammas, betas):

        # Cost Hamiltonian evolution
        for i, j, w in edges:
            qml.IsingZZ(w * gamma, wires=[i, j])
        
        # Mixer Hamiltonian evolution
        for i in range(n):
            qml.RX(2 * beta, wires=i)

In [16]:
# Expected energy from quantum circuit
@qml.qnode(dev)
def qaoa_circuit(params):
    p = len(params) // 2
    gammas = params[:p]
    betas = params[p:]

    ansatz(gammas, betas)
    return qml.expval(H_C)

In [17]:
# Probabilities from quantum circuit
@qml.qnode(dev)
def qaoa_probabilities(params):
    p = len(params) // 2
    gammas = params[:p]
    betas = params[p:]

    ansatz(gammas, betas)
    return qml.probs(wires=range(n))

In [18]:
# Classical optimization of QAOA parameters
p = 1 # Number of QAOA layers

initial = np.concatenate([
    np.full(p, 0.3), # gammas
    np.full(p, 0.4)  # betas
])

result = minimize(
    qaoa_circuit,
    x0=initial,
    method="COBYLA",
    options={"maxiter": 100}
)

In [20]:
# Results
params_opt = result.x
energy_opt = result.fun

print("Optimal gammas:", params_opt[:p])
print("Optimal betas:", params_opt[p:])
print("Minimum energy:", energy_opt)

total_weight = sum(w for i, j, w in edges)
expected_cut = (total_weight - energy_opt) / 2

print("Expected cut value:", expected_cut)

Optimal gammas: [1.7368611]
Optimal betas: [-2.1678823]
Minimum energy: -2.8585271945133486
Expected cut value: 4.429263597256674


In [22]:
# Display the optimized probabilities
probs = qaoa_probabilities(params_opt)

print("\nMeasurement probabilities:")
for index, probability in enumerate(probs):
    bitstring = format(index, f"0{n}b")
    print(f"|{bitstring}>: {probability:.4f}")

# 10. Draw the circuit
print("\nCircuit:")
print(qml.draw(qaoa_circuit)(params_opt))


Measurement probabilities:
|000>: 0.0099
|001>: 0.3414
|010>: 0.0871
|011>: 0.0615
|100>: 0.0615
|101>: 0.0871
|110>: 0.3414
|111>: 0.0099

Circuit:
0: ──H─╭IsingZZ(1.74)────────────────╭IsingZZ(5.21)──RX(-4.34)─┤ ╭<𝓗(2.00,3.00)>
1: ──H─╰IsingZZ(1.74)─╭IsingZZ(3.47)─│───────────────RX(-4.34)─┤ ├<𝓗(2.00,3.00)>
2: ──H────────────────╰IsingZZ(3.47)─╰IsingZZ(5.21)──RX(-4.34)─┤ ╰<𝓗(2.00,3.00)>
